# SIGMOD Exp2 Distinct SNAP Views

Load distinct crossover CSVs and render only the SNAP-included plain and broken-axis views for history and delta sweeps.


In [ ]:
from pathlib import Path
import sys
import importlib
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("../../").resolve()
sys.path.append(str(ROOT / "benches"))
import sigmod_exp_common as _sigmod_exp_common
importlib.reload(_sigmod_exp_common)

from sigmod_exp_common import (
    TOL,
    SIGMOD_BUCKET_NUM,
    SIGMOD_HTAP_TXN_COUNT,
    SIGMOD_HTAP_WAREHOUSE_COUNT,
    SIGMOD_READABLE_EVERY,
    apply_paper_style,
    current_run_stamp,
    ensure_dirs,
)

apply_paper_style(ROOT)

EXP_DIR = (ROOT / "benches" / "sigmod_exp2_distinct_snap_views").resolve()
FIGS_DIR = EXP_DIR / "figs"
ensure_dirs(FIGS_DIR)

CONFIG = {
    "warehouse_count": SIGMOD_HTAP_WAREHOUSE_COUNT,
    "txn_count": SIGMOD_HTAP_TXN_COUNT,
    "bucket_num": SIGMOD_BUCKET_NUM,
    "update_ratio": 0.0001,
    "probe_ratio": 0.01,
    "txn_gc_ratio": 0.05,
    "readable_every": SIGMOD_READABLE_EVERY,
    "repeat": 5,
}

STYLE = {
    ("naive", ""): ("SNAP", TOL["red"], ":", "x"),
    ("ivmh", ""): ("IVMH", TOL["yellow"], "--", "P"),
    ("heap", "Write Repair"): ("MONO-WR", TOL["blue"], "-", "o"),
    ("chain", "Write Repair"): ("DUAL-WR", TOL["cyan"], "-", "s"),
    ("par", "Write Repair"): ("EPOCH-WR", TOL["green"], "-", "D"),
}

RUN_STAMP = current_run_stamp()
CONFIG_TAG = "_".join([
    f"wc{CONFIG['warehouse_count']}",
    f"tc{CONFIG['txn_count']}",
    f"bn{CONFIG['bucket_num']}",
    f"ur{str(CONFIG['update_ratio']).replace('.', 'p')}",
    f"pr{str(CONFIG['probe_ratio']).replace('.', 'p')}",
    f"gc{str(CONFIG['txn_gc_ratio']).replace('.', 'p')}",
    f"re{CONFIG['readable_every']}",
    f"rep{CONFIG['repeat']}",
    "distinct",
])

print("ROOT   :", ROOT)
print("FIGS   :", FIGS_DIR)
print("TAG    :", CONFIG_TAG)


In [ ]:
def candidate_csv_paths(stem):
    names = [
        ROOT / "benches" / "sigmod_data" / f"{stem}_{CONFIG_TAG}.csv",
        ROOT / "benches" / "sigmod_exp2_distinct_crossover" / "data" / f"{stem}_{CONFIG_TAG}.csv",
    ]
    data_dir = ROOT / "benches" / "sigmod_exp2_distinct_crossover" / "data"
    if data_dir.exists():
        names.extend(sorted(data_dir.glob(f"{stem}_*.csv"), reverse=True))
    return names


def find_csv(stem):
    for path in candidate_csv_paths(stem):
        if path.exists():
            return path
    raise FileNotFoundError(f"Could not find CSV for {stem} with tag {CONFIG_TAG}")


def plot_one(ax, df, x_col, xlabel):
    series = [
        ("naive", ""),
        ("ivmh", ""),
        ("heap", "Write Repair"),
        ("chain", "Write Repair"),
        ("par", "Write Repair"),
    ]
    for key in series:
        label, color, linestyle, marker = STYLE[key]
        table_type, repair_type = key
        sub = df[(df["table_type"] == table_type) & (df["repair_type"] == repair_type)].sort_values(x_col)
        if sub.empty:
            continue
        ax.plot(sub[x_col], sub["total_ms"], color=color, linestyle=linestyle, marker=marker, linewidth=1.8, markersize=5, label=label)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Duration (ms / tx)")
    ax.grid(True, linestyle="--", linewidth=0.6, alpha=0.6)
    ax.set_ylim(bottom=0)
    ax.legend(loc="center left", ncol=1, framealpha=0.95)


def snap_break_bounds(df):
    snap = df[df["table_type"] == "naive"]["total_ms"]
    others = df[df["table_type"] != "naive"]["total_ms"]
    if snap.empty or others.empty:
        return None
    lower_max = others.max() * 1.08
    upper_min = snap.min() * 0.92
    if upper_min <= lower_max:
        midpoint = (others.max() + snap.min()) / 2.0
        lower_max = midpoint * 0.95
        upper_min = midpoint * 1.05
    upper_max = snap.max() * 1.03
    return lower_max, upper_min, upper_max


def save_fig(fig, stem):
    stamped = FIGS_DIR / f"{stem}_{RUN_STAMP}.pdf"
    latest = FIGS_DIR / f"{stem}.pdf"
    fig.savefig(stamped, format="pdf", bbox_inches="tight")
    fig.savefig(latest, format="pdf", bbox_inches="tight")
    plt.show()
    print("Saved", stamped)
    print("Saved", latest)


def render_plain(df, x_col, xlabel, stem):
    fig, ax = plt.subplots(1, 1, figsize=(5.0, 4.1))
    plot_one(ax, df, x_col, xlabel)
    fig.tight_layout()
    save_fig(fig, stem)


def render_broken(df, x_col, xlabel, stem):
    bounds = snap_break_bounds(df)
    if bounds is None:
        render_plain(df, x_col, xlabel, stem)
        return
    lower_max, upper_min, upper_max = bounds
    fig, (ax_top, ax_bottom) = plt.subplots(
        2, 1,
        figsize=(5.0, 4.6),
        sharex=True,
        gridspec_kw={"height_ratios": [1.0, 3.0], "hspace": 0.05},
    )
    plot_one(ax_top, df, x_col, "")
    plot_one(ax_bottom, df, x_col, xlabel)
    ax_top.set_ylim(upper_min, upper_max)
    ax_bottom.set_ylim(0, lower_max)
    ax_top.spines["bottom"].set_visible(False)
    ax_bottom.spines["top"].set_visible(False)
    ax_top.tick_params(axis="x", which="both", bottom=False, labelbottom=False)
    ax_top.set_xlabel("")
    ax_top.set_ylabel("")
    if ax_bottom.legend_ is not None:
        ax_bottom.legend_.remove()
    d = 0.012
    kwargs = dict(transform=ax_top.transAxes, color="k", clip_on=False, linewidth=0.8)
    ax_top.plot((-d, +d), (-d, +d), **kwargs)
    ax_top.plot((1 - d, 1 + d), (-d, +d), **kwargs)
    kwargs.update(transform=ax_bottom.transAxes)
    ax_bottom.plot((-d, +d), (1 - d, 1 + d), **kwargs)
    ax_bottom.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)
    fig.tight_layout()
    save_fig(fig, stem)


In [ ]:
history_csv = find_csv("sigmod_exp2_distinct_history")
delta_csv = find_csv("sigmod_exp2_distinct_delta")
print("History CSV:", history_csv)
print("Delta CSV  :", delta_csv)

df_history = pd.read_csv(history_csv, keep_default_na=False)
df_delta = pd.read_csv(delta_csv, keep_default_na=False)
df_history["repair_type"] = df_history["repair_type"].fillna("")
df_delta["repair_type"] = df_delta["repair_type"].fillna("")
df_history["history_pct"] = df_history["history_ratio"] * 100.0
df_delta["delta_pct"] = df_delta["delta_ratio"] * 100.0
display(df_history.head())
display(df_delta.head())


In [ ]:
render_plain(df_history, "history_pct", "Historical Scan Percentage (%)", "exp2-distinct-history-with-snap-view")
render_broken(df_history, "history_pct", "Historical Scan Percentage (%)", "exp2-distinct-history-with-snap-broken-view")
render_plain(df_delta, "delta_pct", "Delta Transaction Percentage (%)", "exp2-distinct-delta-with-snap-view")
render_broken(df_delta, "delta_pct", "Delta Transaction Percentage (%)", "exp2-distinct-delta-with-snap-broken-view")
